# dbt Data Contracts & Model Versioning

A live demo using **dbt-core + DuckDB**. No warehouse required — everything runs locally.

---

## What we'll cover

| Chapter | Topic |
|---------|-------|
| 1 | Raw data — seed tables as the source of truth |
| 2 | Staging layer — transformations without contracts |
| 3 | Data contracts — locking the mart schema |
| 4 | **Contract violation** — what dbt catches and why it matters |
| 5 | **Model versioning** — evolving contracts without breaking consumers |
| 6 | Business analytics — the payoff of a trusted data model |

In [ ]:
import subprocess
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

PROJECT_DIR = Path.cwd()
DB_PATH = PROJECT_DIR / "demo.duckdb"

print(f"Project : {PROJECT_DIR}")
print(f"Database: {DB_PATH}")

In [ ]:
import shutil

# ── Clean slate ────────────────────────────────────────────────────────────
# Remove all dbt artifacts and the database so every run starts fresh.
artifacts = [
    PROJECT_DIR / "demo.duckdb",
    PROJECT_DIR / "demo.duckdb.wal",
    PROJECT_DIR / "target",
    PROJECT_DIR / "logs",
    PROJECT_DIR / "dbt_packages",
]

removed = []
for path in artifacts:
    if path.exists():
        if path.is_dir():
            shutil.rmtree(path)
        else:
            path.unlink()
        removed.append(path.name)

if removed:
    print(f"Removed: {', '.join(removed)}")
else:
    print("Nothing to clean.")
print("Clean slate ready ✓")

## Setup: Build the dbt Pipeline

All artifacts from previous runs (`demo.duckdb`, `target/`, `logs/`) have been removed. Now load the seed data and build all models from scratch.

In [ ]:
def run_dbt(args):
    """Run a dbt command, print concise output, return the result."""
    cmd = ["dbt"] + args + ["--profiles-dir", str(PROJECT_DIR)]
    print(f"$ dbt {' '.join(args)}")
    print("─" * 70)
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(PROJECT_DIR))

    if result.returncode == 0:
        for line in result.stdout.splitlines():
            # Show: model counts, per-model OK lines, timing summary, done
            if any(kw in line for kw in ["Found", " OK ", "Finished running", "Done."]):
                print(line)
        print("\n✅  Completed successfully")
    else:
        print(result.stdout)
        if result.stderr.strip():
            print(result.stderr)

    return result


# Step 1: load CSVs into main_raw schema
run_dbt(["seed"])
print()

# Step 2: build all models (skip the intentionally broken one — that comes in Chapter 4)
run_dbt(["run", "--exclude", "fct_orders_broken"])

In [ ]:
# Open a connection to the database (dbt has already written everything)
con = duckdb.connect(str(DB_PATH))

print("Connected to demo.duckdb ✓\n")
print("All tables in the database:")
con.sql("SHOW ALL TABLES").df()[["database", "schema", "name", "column_names"]]

## Pipeline Architecture

```
┌──────────────────┐    ┌───────────────────────┐    ┌──────────────────────────────────┐
│   main_raw       │    │   main_bronze         │    │   main_serving                   │
│──────────────────│    │───────────────────────│    │──────────────────────────────────│
│ raw_orders       │───▶│ stg_orders  (view)    │─┐  │ fct_orders_v1   ✓ contract       │
│ raw_customers    │───▶│ stg_customers (view)  │ └─▶│ fct_orders_v2   ✓ contract       │
│ raw_products     │───▶│ stg_products  (view)  │───▶│ dim_customers   ✓ contract       │
│                  │    │                       │    │                                  │
│ Seeds (CSV)      │    │ No contracts          │    │ Enforced schema + DB constraints │
└──────────────────┘    └───────────────────────┘    └──────────────────────────────────┘
```

**Contracts live at the serving layer** — the stable interface between data engineering and every downstream consumer (BI tools, ML pipelines, other dbt models).

---
## Chapter 1: Raw Data

Three seed tables loaded from CSV files. This is our source of truth — no transformations, no guarantees, just raw records.

In [ ]:
for table, label in [
    ("main_raw.raw_customers", "raw_customers — 6 rows"),
    ("main_raw.raw_products",  "raw_products — 5 rows"),
    ("main_raw.raw_orders",    "raw_orders — 20 rows"),
]:
    print(f"\n{'━' * 60}")
    print(f"  {label}")
    print(f"{'━' * 60}")
    display(con.sql(f"SELECT * FROM {table}").df())

---
## Chapter 2: The Staging Layer — Transformations, No Contracts

Staging models cast raw types, rename columns, and clean the data. They are **intentionally uncontracted** — this is the fast-iteration zone.

```sql
-- models/bronze/stg_orders.sql
select
    cast(order_id    as bigint)      as order_id,
    cast(customer_id as bigint)      as customer_id,
    cast(product_id  as bigint)      as product_id,
    cast(amount      as decimal(10,2)) as amount,
    status,
    cast(created_at  as date)        as created_at
from {{ source('raw', 'raw_orders') }}
```

No `contract: enforced: true` here. If a developer renames `order_id` to `id` in this model, dbt will happily run it. The problem only surfaces downstream.

In [ ]:
print("stg_orders — column types (no contract, no constraints):")
display(con.sql("DESCRIBE main_bronze.stg_orders").df())

print("\nDatabase-level constraints on stg_orders:")
constraints = con.sql("""
    SELECT constraint_type, constraint_column_names
    FROM duckdb_constraints()
    WHERE table_name = 'stg_orders'
""").df()
if constraints.empty:
    print("  (none — bronze is a view with no enforced constraints)")
else:
    display(constraints)

print("\nFirst 5 rows:")
display(con.sql("SELECT * FROM main_bronze.stg_orders LIMIT 5").df())

---
## Chapter 3: Data Contracts — Locking the Public Interface

At the mart layer, models declare a **data contract**: an explicit promise about column names, types, and constraints.

```yaml
# models/serving/_dim_customers.yml
models:
  - name: dim_customers
    config:
      contract:
        enforced: true          # ← turns on contract enforcement
    columns:
      - name: customer_id
        data_type: bigint       # ← declared type
        constraints:
          - type: not_null      # ← real DB-level NOT NULL
          - type: unique        # ← real DB-level UNIQUE
      - name: name
        data_type: varchar
      - name: email
        data_type: varchar
      - name: country
        data_type: varchar
```

When `enforced: true`, dbt generates a `CREATE TABLE` with **explicit column types and constraints**, then inserts the model's output. If the output doesn't match the declared schema → **build fails immediately**.

In [ ]:
print("dim_customers — column schema (contract enforced):")
display(con.sql("DESCRIBE main_serving.dim_customers").df())

print("\nDatabase-level constraints applied to dim_customers:")
display(
    con.sql("""
        SELECT constraint_type, constraint_column_names
        FROM duckdb_constraints()
        WHERE table_name = 'dim_customers'
    """).df()
)

In [ ]:
print("fct_orders_v2 — column schema (contract enforced):")
display(con.sql("DESCRIBE main_serving.fct_orders_v2").df())

print("\nDatabase-level constraints applied to fct_orders_v2:")
display(
    con.sql("""
        SELECT constraint_type, constraint_column_names
        FROM duckdb_constraints()
        WHERE table_name = 'fct_orders_v2'
    """).df()
)

In [ ]:
print("dim_customers data (6 rows):")
display(con.sql("SELECT * FROM main_serving.dim_customers").df())

print("\nfct_orders_v2 data (first 5 rows):")
display(con.sql("SELECT * FROM main_serving.fct_orders_v2 LIMIT 5").df())

### Under the hood: the generated `CREATE TABLE`

When `contract: enforced: true`, dbt generates a `CREATE TABLE` statement with **explicit types and constraints** baked in, then inserts the model's output into it. Here's the actual DDL dbt compiled for `fct_orders_v2`:

In [ ]:
import re

ddl_path = PROJECT_DIR / "target/run/dbt_contracts_demo/models/serving/fct_orders_v2.sql"
ddl_text = ddl_path.read_text()

# Extract just the CREATE TABLE...;  block (everything up to the first semicolon)
start = ddl_text.lower().find("create")
end = ddl_text.find(";", start) + 1
create_stmt = ddl_text[start:end]

# Clean up Jinja whitespace artifacts
create_stmt = re.sub(r"[ \t]+\n", "\n", create_stmt)   # trailing spaces on lines
create_stmt = re.sub(r"\n{3,}", "\n\n", create_stmt)    # collapse blank lines
create_stmt = re.sub(r"  +", " ", create_stmt)          # collapse double spaces
create_stmt = create_stmt.strip()

print(create_stmt)
print()
print("─" * 70)
print("Every column type and constraint comes directly from the YAML contract.")
print("dbt generated this DDL — the developer never writes it by hand.")

---
## Chapter 4: Contract Violation ⚠️

**The scenario**: A developer refactors `fct_orders_v2`. They rename `order_id` → `id` to align with a new company naming convention. The SQL looks fine. Tests pass locally. They open a PR.

Without contracts, this rename silently deploys and **breaks every downstream consumer** that references `order_id`.

Here's the broken model:

```sql
-- models/serving/fct_orders_broken.sql
-- ⚠️  Intentionally violates the data contract!

select
    o.order_id  as id,          -- ❌ renamed: contract expects 'order_id'
    o.customer_id,
    o.product_id,
    o.amount,
    o.status,
    o.created_at    as order_date,
    p.category      as product_category
from orders o
left join products p on o.product_id = p.product_id
```

Its contract is identical to `fct_orders_v2` — `order_id bigint not null`. Let's see what dbt does when we try to run it.

In [ ]:
print("Running the broken model against its contract...\n")

# DuckDB allows only one writer at a time.
# We close our read connection so dbt can acquire the write lock.
con.close()

cmd = [
    "dbt", "run",
    "--select", "fct_orders_broken",
    "--profiles-dir", str(PROJECT_DIR),
]
result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(PROJECT_DIR))

print(result.stdout)
if result.stderr.strip():
    print(result.stderr)

if result.returncode != 0:
    print("\n" + "═" * 70)
    print("  ❌  dbt REFUSED to build this model")
    print("═" * 70)

# Reopen for subsequent analysis cells
con = duckdb.connect(str(DB_PATH))

### What dbt caught

The error table is the key artifact — dbt shows exactly what differed between the model's output and the contract:

```
| column_name | definition_type | contract_type | mismatch_reason       |
| ----------- | --------------- | ------------- | --------------------- |
| id          | BIGINT          |               | missing in contract   |
| order_id    |                 | BIGINT        | missing in definition |
```

- **`id`** is returned by the model — but it's not declared in the contract
- **`order_id`** is required by the contract — but the model doesn't return it

**Nothing was written to the database.** dbt caught this at compile time, before any SQL ran.

Without contracts this rename would have:
1. ✅ Deployed to production silently
2. ❌ Broken every dashboard, downstream model, and API query referencing `order_id`
3. 🔥 Triggered an incident

**With contracts: caught at build time, not at 2am.**

---
## Chapter 5: Model Versioning — Evolving Contracts Without Breaking Consumers

Contracts solve the "silent breakage" problem, but they create a new challenge: **what if you need to change the schema?**

Adding `product_category` to `fct_orders` is a **backward-incompatible change** for any consumer that `SELECT *`s and maps columns by position. With contracts locked, how do you ship the new column?

The answer is **model versioning**:

```yaml
# models/serving/_fct_orders.yml

models:
  - name: fct_orders
    latest_version: 2         # ref('fct_orders') resolves here by default
    config:
      contract:
        enforced: true        # contracts still enforced on every version

    # Base columns — shared across ALL versions
    columns:
      - name: order_id
        data_type: bigint
        constraints: [not_null]
      - name: customer_id
        data_type: bigint
      # ... (amount, status, order_date)

    versions:
      - v: 1                  # fct_orders_v1: 6 columns — original, stable
      - v: 2                  # fct_orders_v2: 7 columns — adds product_category
        defined_in: fct_orders_v2
        columns:
          - include: all      # inherit all base columns from above
          - name: product_category
            data_type: varchar
```

Both versions are built simultaneously. Consumers choose which to reference.

| Consumer code | Resolves to | Schema |
|---|---|---|
| `ref('fct_orders')` | **v2** (latest_version = 2) | 7 columns incl. `product_category` |
| `ref('fct_orders', version=1)` | v1 | 6 columns — stable forever |
| `ref('fct_orders', version=2)` | v2 | 7 columns |

In [ ]:
v1 = con.sql("SELECT * FROM main_serving.fct_orders_v1 LIMIT 5").df()
v2 = con.sql("SELECT * FROM main_serving.fct_orders_v2 LIMIT 5").df()

print(f"fct_orders  v1  — {len(v1.columns)} columns: {list(v1.columns)}")
display(v1)

print(f"\nfct_orders  v2  — {len(v2.columns)} columns: {list(v2.columns)}")
display(v2)

new_cols = [c for c in v2.columns if c not in v1.columns]
print(f"\nNew in v2: {new_cols}")

In [ ]:
# Both versioned tables coexist in the database right now
rows = []
for tbl, version, note in [
    ("fct_orders_v1", "v1", "baseline — 6 cols, stable"),
    ("fct_orders_v2", "v2 (latest)", "extended — 7 cols, adds product_category"),
    ("dim_customers",  "—", "dimension table"),
]:
    row_count = con.sql(f"SELECT COUNT(*) FROM main_serving.{tbl}").fetchone()[0]
    cols = con.sql(
        f"SELECT column_name FROM information_schema.columns "
        f"WHERE table_name = '{tbl}' AND table_schema = 'main_serving' ORDER BY ordinal_position"
    ).df()["column_name"].tolist()
    rows.append({
        "table": f"main_serving.{tbl}",
        "version": version,
        "rows": row_count,
        "columns": len(cols),
        "column names": ", ".join(cols),
    })

display(pd.DataFrame(rows))

### Migration path for consumers

```
  Legacy consumer                    New consumer
  (pinned to v1)                     (defaults to latest)

  ref('fct_orders', version=1)       ref('fct_orders')
         │                                  │
         ▼                                  ▼
  main_serving.fct_orders_v1        main_serving.fct_orders_v2
  6 columns — stable              7 columns — product_category available
```

When all consumers have migrated to v2, v1 can be **deprecated** (add `deprecation_date`) and eventually **removed** — a controlled, zero-breakage rollout.

---
## Chapter 6: Business Analytics — The Payoff

With `product_category` guaranteed by the v2 contract, downstream analytics can depend on it with **full confidence** — no `COALESCE(product_category, 'Unknown')` defensive coding, no schema drift surprises.

In [ ]:
# Revenue by product category (exclude cancelled orders)
revenue = con.sql("""
    SELECT
        product_category,
        COUNT(*)                    AS orders,
        ROUND(SUM(amount), 2)       AS revenue,
        ROUND(AVG(amount), 2)       AS avg_order_value
    FROM main_serving.fct_orders_v2
    WHERE status != 'cancelled'
    GROUP BY product_category
    ORDER BY revenue DESC
""").df()

print("Revenue by product category (non-cancelled orders):")
display(revenue)

# ── Chart ──────────────────────────────────────────────────────────────────
palette = ["#1565C0", "#F57C00", "#2E7D32"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle(
    "fct_orders_v2  ·  Analytics enabled by the v2 data contract",
    fontsize=12, fontweight="bold", y=1.02
)

# Total revenue
bars = ax1.bar(revenue["product_category"], revenue["revenue"], color=palette, edgecolor="white")
ax1.set_title("Total Revenue by Category", fontweight="bold")
ax1.set_xlabel("Product Category")
ax1.set_ylabel("Revenue (USD)")
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
for bar in bars:
    ax1.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 8,
        f"${bar.get_height():,.0f}", ha="center", va="bottom", fontsize=9, fontweight="bold"
    )
ax1.spines[["top", "right"]].set_visible(False)

# Order count
bars2 = ax2.bar(revenue["product_category"], revenue["orders"], color=palette, edgecolor="white")
ax2.set_title("Order Count by Category", fontweight="bold")
ax2.set_xlabel("Product Category")
ax2.set_ylabel("Orders")
for bar in bars2:
    ax2.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
        int(bar.get_height()), ha="center", va="bottom", fontsize=9, fontweight="bold"
    )
ax2.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig("revenue_by_category.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Order status distribution
status = con.sql("""
    SELECT
        status,
        COUNT(*)                    AS orders,
        ROUND(SUM(amount), 2)       AS revenue
    FROM main_serving.fct_orders_v2
    GROUP BY status
    ORDER BY orders DESC
""").df()

print("Order status breakdown:")
display(status)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
fig.suptitle("Order Status Distribution  ·  fct_orders_v2", fontsize=12, fontweight="bold", y=1.02)

status_palette = {"completed": "#2E7D32", "shipped": "#1565C0", "pending": "#F57C00", "cancelled": "#B71C1C"}
colors = [status_palette.get(s, "#999") for s in status["status"]]

# Order count by status
ax1.bar(status["status"], status["orders"], color=colors, edgecolor="white")
ax1.set_title("Order Count by Status", fontweight="bold")
ax1.set_ylabel("Orders")
for bar in ax1.patches:
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
             int(bar.get_height()), ha="center", va="bottom", fontsize=9, fontweight="bold")
ax1.spines[["top", "right"]].set_visible(False)

# Revenue by status
ax2.bar(status["status"], status["revenue"], color=colors, edgecolor="white")
ax2.set_title("Revenue by Status", fontweight="bold")
ax2.set_ylabel("Revenue (USD)")
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
for bar in ax2.patches:
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
             f"${bar.get_height():,.0f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
ax2.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig("order_status.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Customer revenue summary — joins two contract-enforced tables
customer_summary = con.sql("""
    SELECT
        c.name                       AS customer,
        c.country,
        COUNT(o.order_id)            AS orders,
        ROUND(SUM(o.amount), 2)      AS total_spent,
        ROUND(AVG(o.amount), 2)      AS avg_order
    FROM main_serving.fct_orders_v2 o
    JOIN main_serving.dim_customers c ON o.customer_id = c.customer_id
    GROUP BY c.customer_id, c.name, c.country
    ORDER BY total_spent DESC
""").df()

print("Customer revenue summary (dim_customers ⋈ fct_orders_v2 — both contract-enforced):")
display(
    customer_summary.style.format({
        "total_spent": "${:,.2f}",
        "avg_order": "${:,.2f}",
    })
)

---
## Summary

| Feature | Key takeaway |
|---------|-------------|
| **`contract: enforced: true`** | dbt generates a typed `CREATE TABLE` — column names, types, and constraints are declared in YAML and enforced by the DB |
| **Column constraints** | `not_null`, `unique`, `primary_key` exist at the database level — not just in dbt tests |
| **Contract violation** | Build fails **before any data is written** if the model output doesn't match the declared schema |
| **`latest_version: N`** | `ref('fct_orders')` auto-resolves to the current latest version |
| **Version pinning** | `ref('fct_orders', version=1)` — legacy consumers never break during a schema migration |

---

### The contract enforcement loop

```
Developer changes a model
          │
          ▼
    dbt run --select my_model
          │
    ┌─────┴──────────────────────────────────────────┐
    │  Does model output match the contract?         │
    └─────┬──────────────────────┬───────────────────┘
         YES                     NO
          │                      │
          ▼                      ▼
    Build succeeds         Build FAILS immediately
    Data written           Nothing written to DB
    to the database        Error shows exact column mismatch
```

---

### Try it yourself

```bash
# Build everything (excluding the broken model)
dbt seed && dbt run --exclude fct_orders_broken

# Trigger the contract violation
dbt run --select fct_orders_broken

# Run all models including the broken one to see the full failure
dbt run
```

---
*Stack: dbt-core · dbt-duckdb · DuckDB*